# <center> <img src="../../notebooks/img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Department of Electronics, Systems and Informatics** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Streaming Pipeline — NYC Taxi Trips** </center>
---
**Professor**: Pablo Camarillo Ramirez  
**Student**: Emilio Daniel Guzmán Seda

## Introduction

### Data Description

This pipeline processes synthetic NYC taxi trip records. Each record contains **19 fields** including temporal information (pickup/dropoff), geographic data (LocationIDs), financial details (fare, tips, tolls), and operational metadata (VendorID, payment_type). The Kafka producer injects **dirty data** in a controlled manner:
- ~2% of records with negative or zero `trip_distance`
- ~1% of records with negative or zero `fare_amount`
- ~2% of records with null `PULocationID` or `DOLocationID`
- `congestion_surcharge` and `airport_fee` always `None`
- ~1% duplicate records

### Architecture Diagram

```
┌─────────────────────┐         ┌─────────────────────┐         ┌─────────────────────┐
│                     │         │                     │         │                     │
│  kafka_producer.py  │────────▶│   Apache Kafka      │────────▶│  consumer.ipynb     │
│                     │  JSON   │   (kafka:9093)      │  Stream │  (PySpark SS)       │
│  - Generates trips  │  UTF-8  │   Topic: taxi-trips │         │                     │
│  - Dirty data       │         │                     │         │  foreachBatch:      │
│  - while True loop  │         └─────────────────────┘         │  ┌───────────────┐  │
│                     │                                         │  │ 1. Dedup      │  │
└─────────────────────┘                                         │  │ 2. Clean      │  │
                                                                │  │ 3. Join Zones │  │
                                ┌─────────────────────┐         │  │ 4. Aggregate  │  │
                                │                     │         │  │ 5. Write Neo4j│  │
                                │  taxi+_zone_lookup  │────────▶│  └───────────────┘  │
                                │  .csv (Static DF)   │  Join   │                     │
                                │  data/taxi_zones/   │         └──────────┬──────────┘
                                └─────────────────────┘                    │
                                                                           │ Neo4j Connector
                                                                           ▼
                                                                ┌─────────────────────┐
                                                                │                     │
                                                                │  Neo4j              │
                                                                │  (bolt://neo4j-     │
                                                                │   iteso:7687)       │
                                                                │                     │
                                                                │  (:Borough)         │
                                                                │  (:Trip)            │
                                                                │  [:PICKUP_IN]       │
                                                                │                     │
                                                                └─────────────────────┘
```

### Technology Justification

- **Apache Kafka**: Distributed messaging platform that decouples the producer from the consumer. Offers durability, horizontal scalability, and at-least-once delivery guarantees. Ideal for real-time data ingestion.
- **Spark Structured Streaming**: Stream processing engine that unifies batch and streaming under the same DataFrame API. Enables complex transformations (joins, aggregations, filters) with exactly-once semantics via checkpoints.
- **Neo4j**: Native graph database that allows modeling relationships between entities (trips and boroughs) naturally. The Spark-Neo4j connector enables direct writes from DataFrames using Cypher.

In [1]:
# Install dependencies if needed
# !pip install pyspark kafka-python

In [2]:
# Imports and SparkSession creation
from spark_utils import SparkUtils
from pyspark.sql.functions import col, avg, sum, from_json
from pyspark.sql.types import StructType

# Required packages: Kafka connector + Neo4j connector
packages = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0,org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3"

su = SparkUtils(
    app_name="NYC_Taxi_Streaming_Pipeline",
    master_url="spark://spark-master:7077",
    spark_packages=packages
)

su._spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6ae87674-88a5-446a-9634-86f36eaf1cc6;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	

In [3]:
# Schema definition for the 19 NYC taxi fields
columns_info = [
    ("VendorID", "long"),
    ("tpep_pickup_datetime", "timestamp"),
    ("tpep_dropoff_datetime", "timestamp"),
    ("passenger_count", "long"),
    ("trip_distance", "double"),
    ("RatecodeID", "long"),
    ("store_and_fwd_flag", "string"),
    ("PULocationID", "long"),
    ("DOLocationID", "long"),
    ("payment_type", "long"),
    ("fare_amount", "double"),
    ("extra", "double"),
    ("mta_tax", "double"),
    ("tip_amount", "double"),
    ("tolls_amount", "double"),
    ("improvement_surcharge", "double"),
    ("total_amount", "double"),
    ("congestion_surcharge", "double"),
    ("airport_fee", "double")
]

# Generate StructType schema from columns_info
taxi_schema = SparkUtils.generate_schema(columns_info)
taxi_schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True)])

In [4]:
# Load static taxi zones DataFrame
zones_df = su._spark.read.csv(
    "/opt/spark/work-dir/data/taxi_zones/taxi+_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zones_df.show(5)
zones_df.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



## Phase 1: Data Ingestion from Kafka

In [5]:
# Read stream from Kafka and parse JSON
kafka_df = (su._spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9093")
    .option("subscribe", "taxi-trips")
    .load())

# Convert Kafka binary value to JSON string
json_df = kafka_df.selectExpr("CAST(value AS STRING) as json_str")

# Parse JSON using the defined schema
df_parsed = (json_df
    .select(from_json(col("json_str"), taxi_schema).alias("data"))
    .select("data.*"))

df_parsed.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



## Phase 2: foreachBatch Function — Cleaning, Enrichment and Persistence

In [6]:
# Neo4j connection configuration
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

neo4j_options = {
    "url": neo4j_url,
    "authentication.basic.username": neo4j_user,
    "authentication.basic.password": neo4j_passwd,
    "batch.size": "5000",
    "transaction.retries": "3"
}


def process_micro_batch(batch_df, batch_id):
    """Processes each micro-batch: cleaning, enrichment, aggregation and Neo4j write."""
    # 1. Check for empty batch
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: empty, skipping...")
        return

    print(f"--- Batch {batch_id}: {batch_df.count()} records received ---")

    # 2. Deduplication
    df_clean = batch_df.dropDuplicates()

    # 3. Drop unnecessary columns
    df_clean = df_clean.drop("congestion_surcharge", "airport_fee")

    # 4. Drop nulls in critical fields
    df_clean = df_clean.dropna(subset=["PULocationID", "DOLocationID", "fare_amount"])

    # 5. Filter invalid records
    df_clean = df_clean.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))

    if df_clean.isEmpty():
        print(f"Batch {batch_id}: no valid records after cleaning")
        return

    # 6. Stream-Static Join with zones
    df_enriched = df_clean.join(zones_df, df_clean.PULocationID == zones_df.LocationID, "left") \
                          .withColumnRenamed("Borough", "Pickup_Borough") \
                          .drop("LocationID", "Zone", "service_zone")

    # 7. Filter records without borough (no match in join)
    df_with_borough = df_enriched.filter(col("Pickup_Borough").isNotNull())

    # 8. Write Borough nodes to Neo4j
    borough_df = df_with_borough.select(col("Pickup_Borough").alias("name")).distinct()
    borough_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Overwrite") \
        .options(**neo4j_options) \
        .option("labels", ":Borough") \
        .option("node.keys", "name") \
        .save()

    # 9. Write Trip nodes to Neo4j
    trip_df = df_with_borough.select(
        col("VendorID").alias("vendor_id"),
        col("tpep_pickup_datetime").cast("string").alias("pickup_dt"),
        col("tpep_dropoff_datetime").cast("string").alias("dropoff_dt"),
        "passenger_count", "trip_distance", "fare_amount",
        "tip_amount", "total_amount", "payment_type",
        col("Pickup_Borough").alias("pickup_borough")
    )
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("labels", ":Trip") \
        .option("node.keys", "vendor_id,pickup_dt,pickup_borough") \
        .save()

    # 10. Write PICKUP_IN relationships using Cypher query
    rel_query = """
    MATCH (t:Trip {vendor_id: event.vendor_id, pickup_dt: event.pickup_dt, pickup_borough: event.pickup_borough})
    MATCH (b:Borough {name: event.pickup_borough})
    MERGE (t)-[:PICKUP_IN]->(b)
    """
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("query", rel_query) \
        .save()

    # 11. Calculate and display aggregations
    df_agg = df_with_borough.groupBy("Pickup_Borough", "VendorID", "payment_type") \
        .agg(
            avg("tip_amount").alias("avg_tip"),
            sum("total_amount").alias("total_revenue")
        )
    print(f"Batch {batch_id}: Aggregations computed")
    df_agg.show(10, truncate=False)

## Phase 3: Stream Execution

In [7]:
from pathlib import Path
import shutil

# Clean checkpoint directory before starting (class pattern)
checkpoint_path = "/opt/spark/work-dir/checkpoints/taxi_streaming_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Start the stream with foreachBatch
query = (df_parsed.writeStream
    .foreachBatch(process_micro_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start())

print("Streaming pipeline started. Press Ctrl+C to stop.\n")
su._spark.streams.awaitAnyTermination()

26/05/12 00:21:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming pipeline started. Press Ctrl+C to stop.

Batch 0: empty, skipping...


--- Batch 1: 1 records received ---
Batch 1: no valid records after cleaning


--- Batch 2: 4 records received ---


Batch 2: Aggregations computed
+--------------+--------+------------+-------+-------------+
|Pickup_Borough|VendorID|payment_type|avg_tip|total_revenue|
+--------------+--------+------------+-------+-------------+
|Manhattan     |2       |1           |18.51  |90.41        |
|Brooklyn      |2       |4           |12.77  |169.14       |
|Brooklyn      |1       |2           |9.16   |47.07        |
|Queens        |2       |4           |4.73   |77.27        |
+--------------+--------+------------+-------+-------------+



...

Batch 50: Aggregations computed
+--------------+--------+------------+-------+-------------+
|Pickup_Borough|VendorID|payment_type|avg_tip|total_revenue|
+--------------+--------+------------+-------+-------------+
|Manhattan     |1       |1           |5.85   |54.14        |
|Brooklyn      |2       |1           |12.345 |270.78       |
|Bronx         |1       |1           |1.53   |38.91        |
|Staten Island |1       |1           |11.8   |153.58       |
+--------------+--------+------------+-------+-------------+

--- Batch 51: 5 records received ---


Batch 51: Aggregations computed
+--------------+--------+------------+-------+-------------+
|Pickup_Borough|VendorID|payment_type|avg_tip|total_revenue|
+--------------+--------+------------+-------+-------------+
|Queens        |2       |1           |19.49  |125.91       |
|Manhattan     |2       |3           |1.27   |115.28       |
|Queens        |1       |4           |12.12  |132.56       |
|Bronx         |2       |3           |16.38  |108.83       |
|Brooklyn      |2       |3           |1.7    |99.67        |
+--------------+--------+------------+-------+-------------+



--- Batch 52: 5 records received ---


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
                                                                                

KeyboardInterrupt: 

In [ ]:
su.spark.stop()